# fMRI Reliability Pipeline
- Designed to parametrically analyze fMRI files to identify which pipelines produce the most robust results.

In [ ]:
import pandas as pd
import itertools
import numpy as np

from scipy.spatial.distance import pdist
from scipy.stats import f_oneway, ttest_ind, ttest_rel
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Plotting Packages 
import seaborn as sns
import matplotlib.pyplot as plt
import ptitprince as pt # Only needed for raincloud plot
import configFunctions as config
from nilearn import plotting

### Required files
- 'coordinatesAll.csv' - generated by serverFunctions, final output of mergeOutputs.sh.

In [ ]:
# A configuration function to make output images easier to modify for publication.
config.svg_editing()

In [ ]:
# load data, inspect
df = pd.read_csv('coordinatesAll.csv')
df.head()

### Intermediate clean up

In [ ]:
# Pull out and show the rows which include any nans
rows_with_nan = df[df.isna().any(axis=1)]
rows_with_nan.to_csv('coordinatesAll_NaNRows.csv', index=False)
subj_with_nan = rows_with_nan['subject'].unique()
print(f'Subjects with NaNs - {len(subj_with_nan)}')
print(f'List: {subj_with_nan}')
print(f'Dataframe filtered to remove subjects with any NaNs')

df = df[~df['subject'].isin(subj_with_nan)]
# ['107','103','99','45','83','108','110','88','109','115','111','15','116','112','104','100','117','105','101','118','102']
print(f'{len(df['subject'].unique())} Subjects Remaining...')


In [ ]:
# Group by subject and condition
grouping_vars = ['subject', 'target', 'clustMask', 'sequence_type']

subjVec = df['subject'].unique()
targetVec = df['target'].unique()
cMaskVec = df['clustMask'].unique()
seqTypeVec = df['sequence_type'].unique()

# Create an array with each possible combo.
combinations = list(itertools.product(subjVec, targetVec, cMaskVec, seqTypeVec))

# Put into pandas dataframe
combodf = pd.DataFrame(combinations, columns=['subject', 'target', 'clustMask', 'sequence_type'])
combodf.head()

In [ ]:
# Perform the analysis - span the set of each combination of parameters, take the mean euclidean distance across those sessions.
results = []

for combo in combinations:
    df_filt = df[
        (df['subject'] == combo[0]) & 
        (df['target'] == combo[1]) & 
        (df['clustMask'] == combo[2]) & 
        (df['sequence_type'] == combo[3])
    ]
    meanDist = round(pdist(df_filt[['X','Y','Z']], metric='euclidean').mean(),2)
    results.append(meanDist)

# Append this new column to the previously generated df.
combodf['mean_distance'] = results
combodf.head()

# Plotting

In [ ]:
groupVars = ['target', 'clustMask', 'sequence_type'] # Matching df column names
targetName = 'sgACC' # For plotting, what does the 'target' mask represent?
clusterRegion = 'dlPFC' # for plotting, what does the 'clusterMask' mask represent?
groupVarNames = [f'{targetName} mask', f'{clusterRegion} mask', 'Sequence Type'] # The names for each of the above groupVars for plotting purposes

# For nilearn brain views, to visualize the output coordinates of processing with the clustMask type mentioned below, must match exactly.
colorDict = {
    'orig': 'red',
    'dilate_5': 'green',
    'erode_1': 'blue'
}


In [ ]:
def plot_significance_brackets(ax, comparisons, y_min, y_max, test_type='anova'):
    """
    Plot significance brackets and stars on an axis.
    
    Parameters:
    -----------
    ax : matplotlib axis
        The axis to plot on
    comparisons : list of dict
        List of comparisons, each containing 'pos1', 'pos2', and 'pval'
    y_min : float
        Minimum y-value in the data
    y_max : float
        Maximum y-value in the data
    test_type : str
        Either 'ttest' or 'anova' to determine bracket style
    
    Returns:
    --------
    tuple : (new_y_min, new_y_max) adjusted limits to accommodate brackets
    """
    y_range = y_max - y_min
    
    if test_type == 'ttest':
        # For t-test (single comparison between 2 groups)
        comp = comparisons[0]
        pval = comp['pval']
        
        # Add significance line and star
        line_height = y_max + 0 * y_range
        star_height = line_height + 0.01 * y_range
        
        # Draw bracket line
        ax.plot([0, 1], [line_height, line_height], 'k-', linewidth=1.5)
        ax.plot([0, 0], [line_height - 0.02 * y_range, line_height], 'k-', linewidth=1.5)
        ax.plot([1, 1], [line_height - 0.02 * y_range, line_height], 'k-', linewidth=1.5)
        
        # Add star based on significance level
        if pval < 0.001:
            star = '***'
        elif pval < 0.01:
            star = '**'
        elif pval < 0.05:
            star = '*'
        else:
            star = 'ns'
        
        ax.text(0.5, star_height, star, ha='center', va='bottom', 
                fontsize=16, fontweight='bold')
        
        # Return adjusted y-limits
        return (y_min - 0.05 * y_range, star_height + 0.05 * y_range)
        
    else:  # ANOVA with multiple comparisons
        if not comparisons:
            # No significant comparisons, return original limits
            return (y_min, y_max)
            
        # Draw brackets for significant comparisons
        bracket_height_start = y_max + 0.02 * y_range
        bracket_spacing = 0.08 * y_range
        
        for comp_idx, comp in enumerate(comparisons):
            x1 = comp['pos1']
            x2 = comp['pos2']
            pval = comp['pval']
            
            # Determine star based on p-value
            if pval < 0.001:
                star = '***'
            elif pval < 0.01:
                star = '**'
            elif pval < 0.05:
                star = '*'
            else:
                star = 'ns'
            
            # Calculate bracket height (stack them)
            bracket_y = bracket_height_start + (comp_idx * bracket_spacing)
            star_y = bracket_y + 0.005 * y_range
            
            # Draw horizontal line
            ax.plot([x1, x2], [bracket_y, bracket_y], 'k-', linewidth=1.5)
            
            # Draw vertical ticks on both ends
            tick_height = 0.015 * y_range
            ax.plot([x1, x1], [bracket_y - tick_height, bracket_y], 'k-', linewidth=1.5)
            ax.plot([x2, x2], [bracket_y - tick_height, bracket_y], 'k-', linewidth=1.5)
            
            # Add star
            ax.text((x1 + x2) / 2, star_y, star, ha='center', va='bottom', 
                    fontsize=14, fontweight='bold')
        
        # Return adjusted y-limits to accommodate all brackets
        max_bracket_y = bracket_height_start + (len(comparisons) - 1) * bracket_spacing
        return (y_min - 0.05 * y_range, max_bracket_y + 0.1 * y_range)

### Single Subject plot

In [ ]:
df_subj = df[(df['subject'] == 3)].copy()
df_subj['colors'] = df_subj['clustMask'].map(colorDict)

plot_coords = np.array(df_subj[['X','Y','Z']])
plot_colors = list(df_subj['colors'])
view = plotting.view_markers(plot_coords, plot_colors, marker_size=5)

# Viewing options
# view.open_in_browser()
view

### Plot the mean coordinates for each subject

In [ ]:
df_subj_centroid = df.groupby('subject')[['X', 'Y', 'Z']].mean()

df_subj['colors'] = df_subj['clustMask'].map(colorDict)

plot_coords = np.array(df_subj_centroid[['X','Y','Z']])
# plot_colors = list(df_subj['colors'])
view = plotting.view_markers(plot_coords, 'red', marker_size=5)
# view.open_in_browser()
view

### Statistical Testing

In [ ]:
# Perform all statistical tests before plotting
# This ensures consistency between tests and visualizations

# Define the group ordering (must match plotting cell)
orderVec = [['seed', 'network'],
            ['erode_1', 'orig', 'dilate_5'],
            ['se_e2', 'se', 'me']]

# Storage for statistical results
stat_results = {}

for idx, var in enumerate(groupVars):
    # Convert to categorical with specified order
    combodf_test = combodf.copy()
    combodf_test[var] = pd.Categorical(combodf_test[var], categories=orderVec[idx], ordered=True)
    
    # Get groups in the correct order
    groups = combodf_test[var].cat.categories.tolist()
    group_data = [combodf_test[combodf_test[var] == val]['mean_distance'].dropna()
                  for val in groups]
    
    # Perform statistical test (t-test for 2 groups, ANOVA for more)
    if len(groups) == 2:
        # calculate and report the difference between the two groups
        mean_diff = group_data[0].mean() - group_data[1].mean()
        print(f'Difference between {groups[0]} and {groups[1]}: {mean_diff}')

        # Perform t-test
        stat, p_val = ttest_rel(group_data[0], group_data[1])
        stat_name = 't'
        
        print(f'Between {groups} {stat_name} test')
        print(f'Stat {stat}')
        print(f'Stat {p_val}')
        
        # Store single comparison result
        stat_results[var] = {
            'test': 'ttest',
            'stat': stat,
            'p_val': p_val,
            'stat_name': stat_name,
            'groups': groups,
            'comparisons': [
                {
                    'group1': groups[0],
                    'group2': groups[1],
                    'pval': p_val,
                    'pos1': 0,
                    'pos2': 1
                }
            ]
        }
    else:
        # ANOVA for more than 2 groups
        stat, p_val = f_oneway(*group_data)
        stat_name = 'F'
        print(f'Among {groups} ANOVA test')
        print(f'Stat {stat}')
        print(f'P-val {p_val}')
        
        # Perform post-hoc Tukey HSD test
        print(f'\n{groupVarNames[idx]} - Post-hoc Tukey HSD Test:')
        print('=' * 60)
        
        tukey_data = combodf_test[[var, 'mean_distance']].dropna()
        tukey_result = pairwise_tukeyhsd(endog=tukey_data['mean_distance'], 
                                         groups=tukey_data[var], 
                                         alpha=0.05)
        print(tukey_result)
        print()
        
        # Convert Tukey results to DataFrame for robust column access
        tukey_summary = tukey_result.summary()
        tukey_df = pd.DataFrame(data=tukey_summary.data[1:], columns=tukey_summary.data[0])
        
        # Extract all comparisons from Tukey results using named columns
        comparisons = []
        group_positions = {group: i for i, group in enumerate(groups)}
        
        for _, row in tukey_df.iterrows():
            group1 = row['group1']
            group2 = row['group2']
            reject = row['reject']  # Boolean for whether to reject null hypothesis
            pval = row['p-adj']     # Adjusted p-value
            
            if reject:  # Only store significant comparisons
                comparisons.append({
                    'group1': group1,
                    'group2': group2,
                    'pval': pval,
                    'pos1': group_positions[group1],
                    'pos2': group_positions[group2]
                })
        
        # Sort comparisons by distance between groups (shortest first)
        comparisons.sort(key=lambda x: abs(x['pos2'] - x['pos1']))
        
        # Store results
        stat_results[var] = {
            'test': 'anova',
            'stat': stat,
            'p_val': p_val,
            'stat_name': stat_name,
            'groups': groups,
            'comparisons': comparisons,
            'tukey_result': tukey_result
        }

print('\n' + '='*60)
print('Statistical analysis complete. Results stored for plotting.')
print('='*60)

### Raincloud plot - half violin, box, and rainplot.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Define your desired order (must match statistical analysis cell)
orderVec = [['seed', 'network'],
            ['erode_1', 'orig', 'dilate_5'],
            ['se_e2', 'se', 'me']]
nameVec = [['Seed', 'Network'],  
           ['Eroded', 'Original', 'Dilated'],  # Display names for second plot
           ['Single Echo', 'Multi-Echo', 'Multi-Echo + ICA']]  # Display names for third plot

for idx, (var, varPlot) in enumerate(zip(groupVars, groupVarNames)):
    # Convert the column to categorical with specified order
    combodf[var] = pd.Categorical(combodf[var], categories=orderVec[idx], ordered=True)

    testVar = pt.RainCloud(data=combodf, x=var, y='mean_distance', ax=axes[idx], 
        palette='Set2', hue=var, bw=0.3, width_viol=0.7, width_box=0.3,
        orient='v', alpha=0.6, dodge=False, pointplot=False, point_size=3, 
        linewidth=2, jitter=0.15)
    
    axes[idx].set_xticklabels(nameVec[idx])
    
    # Open up the plot to ensure the peaks for each distribution are visible.
    percSpread = 0.4 # Defines how much you want to extend past current limits.
    xmin, xmax = axes[idx].get_xlim()
    axes[idx].set_xlim((xmin-(percSpread), xmax+(percSpread)))

    # Get statistical results from the analysis cell
    results = stat_results[var]
    stat = results['stat']
    p_val = results['p_val']
    stat_name = results['stat_name']
    comparisons = results['comparisons']
    
    # Get y-axis bounds
    y_max = combodf['mean_distance'].max()
    y_min = combodf['mean_distance'].min()
    
    # Plot significance brackets and get adjusted y-limits
    new_y_min, new_y_max = plot_significance_brackets(
        ax=axes[idx],
        comparisons=comparisons,
        y_min=y_min,
        y_max=y_max,
        test_type=results['test']
    )
    
    # Apply adjusted y-limits
    axes[idx].set_ylim(new_y_min, new_y_max)
    
    # Add title with statistics
    axes[idx].set_title(f'{varPlot}\n{stat_name}={stat:.2f}, p={p_val:.4f}',
                        fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('', fontsize=11)
    axes[idx].set_ylabel('Mean Pairwise Distance (mm)', fontsize=11)

plt.tight_layout()
plt.savefig('Rainplot.svg')
plt.show()